# PCA Eigen-Portfolio + GenAI Factor Intelligence

This notebook builds a classic **eigen-portfolio** (PCA on standardized equity returns), then layers on three GenAI components so the output is actually *readable* instead of a matrix of loadings:

1. **LLM Component Interpreter** — auto-labels each principal component in plain English (e.g. "broad market factor", "India cyclicals vs. defensives").
2. **RAG News Explainer** — retrieves real headlines around the biggest factor moves and asks an LLM to explain *why*, grounded only in retrieved evidence.
3. **Auto Report Generator** — compiles everything (variance explained, interpretations, anomaly narratives) into a markdown research report with an LLM-written executive summary.

Built for Google Colab. You'll need an Anthropic API key (get one at console.anthropic.com) — it's requested via `getpass` so it's never hardcoded or saved in the notebook.

In [ ]:
!pip -q install yfinance langchain langchain-anthropic langchain-community langchain-huggingface faiss-cpu sentence-transformers feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import getpass
from sklearn.decomposition import PCA

## 1. Data — universe, download, intraday returns
(Fixed: removed duplicate `RELIANCE.NS`, `.append()` replaced with `pd.concat` since `.append()` was removed in pandas 2.x.)

In [ ]:
name = ['GOOG','AMZN','SBIN.BO','MSFT','LT.NS','RELIANCE.NS','TATASTEEL.NS','HDFCLIFE.NS',
        'ITC.NS','MARUTI.NS','COALINDIA.NS','BHARTIARTL.NS','NTPC.NS','TATACONSUM.NS',
        'KOTAKBANK.NS','NESTLEIND.NS','BRITANNIA.NS','BAJFINANCE.NS','BAJAJFINSV.NS']

In [ ]:
def dataset(name):
    # Build each ticker as a DATE-INDEXED Series, then join on actual dates.
    # This matters here because we're mixing NYSE tickers (GOOG, AMZN, MSFT)
    # with NSE/BSE tickers (.NS / .BO) - different holiday calendars mean each
    # ticker can have a different number of trading days. Joining positionally
    # (like the original code did) silently misaligns dates across markets.
    series_list = []
    for i in name:
        y = yf.Ticker(i)
        d = y.history(interval='1d', start='2024-01-01', end='2024-03-01')
        d.index = d.index.tz_localize(None).normalize()  # drop tz + time so dates line up across exchanges
        intraday_ret = (d['Close'] - d['Open']) * 100 / d['Open']
        intraday_ret.name = i
        series_list.append(intraday_ret)
    # inner join: keep only dates where EVERY market in the universe was open
    y_f = pd.concat(series_list, axis=1, join='inner')
    return y_f, y_f.index.tolist()

## 2. Normalize and split train/test

In [ ]:
asset_returns = y_f.pct_change(1).dropna()
normed_returns = (asset_returns - asset_returns.mean()) / (asset_returns.std() + 1e-8)
normed_returns = normed_returns.dropna(axis=1)

train_end = '2024-02-15'
df_train = normed_returns[normed_returns.index <= train_end].copy()
df_test  = normed_returns[normed_returns.index > train_end].copy()
df_raw_train = asset_returns[asset_returns.index <= train_end].copy()
df_raw_test  = asset_returns[asset_returns.index > train_end].copy()

print('Train dataset:', df_train.shape)
print('Test dataset:', df_test.shape)
assert df_train.shape[0] > 0, 'df_train is empty - check date alignment above'

## 3. PCA — fit the eigen-portfolios
(Fixed: `df_pca` now keeps the actual dates as its index instead of a default RangeIndex — needed later so we can match component spikes to calendar dates.)

In [ ]:
pca = PCA(n_components=7)
fit = pca.fit_transform(df_train)
df_pca = pd.DataFrame(fit, columns=np.arange(pca.n_components_), index=df_train.index)

print('Variance explained per component:')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {v:.1%}')
print(f'Cumulative: {pca.explained_variance_ratio_.sum():.1%}')
df_pca.head()

## 4. Sharpe ratio helper

In [ ]:
def sharpe_ratio(ts_returns, periods_per_year=252):
    n_years = ts_returns.shape[0] / periods_per_year
    annualized_return = np.power(np.prod(1 + ts_returns), (1 / n_years)) - 1
    annualized_vol = ts_returns.std() * np.sqrt(periods_per_year)
    annualized_sharpe = annualized_return / annualized_vol
    return annualized_return, annualized_vol, annualized_sharpe

for i in range(pca.n_components_):
    ar, av, sh = sharpe_ratio(df_pca[i])
    print(f'PC{i+1}:  return={ar:.2%}  vol={av:.2%}  sharpe={sh:.2f}')

---
# GenAI Extensions

Each section below follows the same four-part structure: **define the LLM -> define the pipeline -> build the chain -> run the chain.**

In [ ]:
GOOGLE_API_KEY = getpass.getpass('Enter your Google AI Studio API key: ')

!pip -q install langchain-google-genai langchain-text-splitters

## 5. LLM Component Interpreter

In [ ]:
# ============================================================
# 1. DEFINE THE LLM
# ============================================================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash', temperature=0.3, google_api_key=GOOGLE_API_KEY)

# Gemini returns content as a list of blocks (+ signature metadata) instead of a plain
# string. This helper extracts just the text - used across all chains below.
def get_text(response):
    content = response.content
    if isinstance(content, list):
        return ''.join(block.get('text', '') for block in content if isinstance(block, dict))
    return content


# ============================================================
# 2. DEFINE THE PIPELINE (data prep + prompt template)
# ============================================================
def get_top_loadings(pca, feature_names, component_idx, n=5):
    loadings = pca.components_[component_idx]
    order = np.argsort(loadings)
    top_neg = [(feature_names[i], loadings[i]) for i in order[:n]]
    top_pos = [(feature_names[i], loadings[i]) for i in order[::-1][:n]]
    return top_pos, top_neg

interpret_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a quantitative portfolio analyst. You explain PCA-derived eigen-portfolios '
     'in clear, precise language for a research report. Given the top positively- and '
     'negatively-weighted stocks for a principal component, propose a short label (3-6 words) '
     'for the market factor it likely represents, then explain your reasoning in 2-3 sentences. '
     'Be specific about sectors/geographies when the tickers suggest a pattern. If unclear, say so '
     'honestly rather than overclaiming.'),
    ('human',
     'Component {idx} explains {variance:.1%} of variance.\n\n'
     'Top POSITIVE loadings:\n{pos}\n\nTop NEGATIVE loadings:\n{neg}\n\n'
     'Label and explain this component.')
])


# ============================================================
# 3. BUILD THE CHAIN
# ============================================================
interpret_chain = interpret_prompt | llm | RunnableLambda(get_text)


# ============================================================
# 4. RUN THE CHAIN
# ============================================================
import time

feature_names = df_train.columns.tolist()
interpretations = {}
for i in range(pca.n_components_):
    pos, neg = get_top_loadings(pca, feature_names, i)
    pos_str = '\n'.join(f'  {t}: {w:+.3f}' for t, w in pos)
    neg_str = '\n'.join(f'  {t}: {w:+.3f}' for t, w in neg)
    try:
        interpretations[i] = interpret_chain.invoke({
            'idx': i + 1, 'variance': pca.explained_variance_ratio_[i],
            'pos': pos_str, 'neg': neg_str
        })
    except Exception as e:
        print(f'PC{i+1} failed: {e}')
        interpretations[i] = f'(interpretation unavailable: {e})'
    print(f'--- PC{i+1} ({pca.explained_variance_ratio_[i]:.1%} variance) ---')
    print(interpretations[i], '\n')
    time.sleep(4)  # stay under free-tier rate limits

## 6. RAG News Explainer

In [ ]:
# ============================================================
# 1. DEFINE THE LLM
# ============================================================
# Reuses `llm` and `get_text` defined in Section 5.


# ============================================================
# 2. DEFINE THE PIPELINE (retrieval: fetch -> chunk -> embed -> store)
# ============================================================
import feedparser
from urllib.parse import quote
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# Headlines are short, but a splitter is included so this pipeline still works
# correctly if you later feed it longer documents (full articles, filings, etc.)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

def fetch_news_rss(query, max_items=5):
    url = f'https://news.google.com/rss/search?q={quote(query)}&hl=en-IN&gl=IN&ceid=IN:en'
    feed = feedparser.parse(url)
    return [f"{e.title}. {e.get('summary','')}" for e in feed.entries[:max_items]]

def build_retriever(raw_docs, k=4):
    chunks = splitter.create_documents(raw_docs)
    vs = FAISS.from_documents(chunks, embeddings)
    return vs.as_retriever(search_kwargs={'k': k})

rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a markets analyst writing a short factor-attribution note. Use ONLY the '
     'retrieved headlines given as context - do not invent facts. If the headlines do not '
     'clearly explain the move, say plainly that the cause is unclear from available news.'),
    ('human',
     'On {date}, eigen-portfolio Component {comp} moved sharply, driven mainly by: {top_stocks}.\n\n'
     'Retrieved headlines from around that date:\n{context}\n\n'
     'In 3-4 sentences, give a plausible explanation for this move, citing the headlines.')
])


# ============================================================
# 3. BUILD THE CHAIN
# ============================================================
rag_generation_chain = rag_prompt | llm | RunnableLambda(get_text)

def run_rag_chain(date, comp_idx, top_k=4):
    pos, neg = get_top_loadings(pca, feature_names, comp_idx, n=3)
    drivers = pos + neg
    top_stocks_str = ', '.join(t for t, _ in drivers)
    queries = [t.split('.')[0] for t, _ in drivers[:3]]

    raw_docs = []
    for q in queries:
        raw_docs += fetch_news_rss(f'{q} stock')
    if not raw_docs:
        return 'No relevant news retrieved for this date.'

    retriever = build_retriever(raw_docs, k=top_k)
    retrieved = retriever.invoke(f'{top_stocks_str} stock news')
    context = '\n'.join(f'- {d.page_content}' for d in retrieved)

    return rag_generation_chain.invoke({
        'date': date, 'comp': comp_idx + 1, 'top_stocks': top_stocks_str, 'context': context
    })


# ============================================================
# 4. RUN THE CHAIN
# ============================================================
def top_anomaly_dates(component_idx, top_n=2):
    col = df_pca.iloc[:, component_idx]
    return col.abs().sort_values(ascending=False).head(top_n).index.tolist()

anomaly_narratives = []
for comp_idx in range(3):  # top 3 components, 2 anomaly dates each
    for date in top_anomaly_dates(comp_idx, top_n=2):
        narrative = run_rag_chain(date, comp_idx)
        date_str = str(date.date()) if hasattr(date, 'date') else str(date)
        anomaly_narratives.append((date_str, comp_idx, narrative))
        print(f'--- PC{comp_idx+1} on {date_str} ---')
        print(narrative, '\n')
        time.sleep(4)

## 7. Auto Report Generation

In [ ]:
# ============================================================
# 1. DEFINE THE LLM
# ============================================================
# Reuses `llm` and `get_text` from Section 5.


# ============================================================
# 2. DEFINE THE PIPELINE (report assembly + summary prompt)
# ============================================================
def build_report_sections():
    sections = []
    sections.append(f'# Eigen-Portfolio Factor Report\nGenerated {datetime.date.today()}\n')
    sections.append(
        f'## Variance Explained\nTop {pca.n_components_} components explain '
        f'{pca.explained_variance_ratio_.sum():.1%} of total variance in the universe.\n')

    sections.append('## Component Interpretations\n')
    for i in range(pca.n_components_):
        sections.append(f'### PC{i+1} ({pca.explained_variance_ratio_[i]:.1%} variance)\n{interpretations[i]}\n')

    sections.append('## Anomaly Explanations (RAG-grounded)\n')
    for date, comp, narrative in anomaly_narratives:
        sections.append(f'**{date} — PC{comp+1}:** {narrative}\n')

    return sections

exec_summary_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You write concise executive summaries for quant research reports.'),
    ('human', 'Summarize this factor report in one tight paragraph for a portfolio manager:\n\n{report}')
])


# ============================================================
# 3. BUILD THE CHAIN
# ============================================================
summary_chain = exec_summary_prompt | llm | RunnableLambda(get_text)


# ============================================================
# 4. RUN THE CHAIN
# ============================================================
report_sections = build_report_sections()
exec_summary = summary_chain.invoke({'report': '\n'.join(report_sections)})

final_report = f'# Executive Summary\n{exec_summary}\n\n' + '\n'.join(report_sections)

with open('eigen_portfolio_report.md', 'w') as f:
    f.write(final_report)

print(final_report[:1500], '...\n\n[full report saved to eigen_portfolio_report.md]')

### Optional: download the report from Colab
```python
from google.colab import files
files.download('eigen_portfolio_report.md')
```
You can paste the `.md` into a PDF exporter (e.g. Pandoc, or `md-to-pdf`) if you want a shareable PDF.